In [1]:
%pip install torch transformers datasets evaluate tqdm

%pip install -q "transformers[sentencepiece]" datasets accelerate peft gdown bitsandbytes

%pip install deepspeed

%pip install evaluate

%pip install huggingface_hub[hf_xet]


You should consider upgrading via the 'c:\Users\Burhanuddin\Home\gec_pipeline\.venv\Scripts\python.exe -m pip install --upgrade pip' command.


Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\Burhanuddin\Home\gec_pipeline\.venv\Scripts\python.exe -m pip install --upgrade pip' command.


You should consider upgrading via the 'c:\Users\Burhanuddin\Home\gec_pipeline\.venv\Scripts\python.exe -m pip install --upgrade pip' command.



Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\Burhanuddin\Home\gec_pipeline\.venv\Scripts\python.exe -m pip install --upgrade pip' command.


Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\Burhanuddin\Home\gec_pipeline\.venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [2]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForCausalLM
)
from peft import PeftModel
from tqdm import tqdm
import json
import os


c:\Users\Burhanuddin\Home\gec_pipeline\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def load_data(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    incorrect = [item["incorrect_pair"] for item in data]
    correct = [item["correct_pair"] for item in data]
    return incorrect, correct


In [4]:
def load_model_and_tokenizer(base=None, checkpoint_dir=None, model_type=None):
    if base:
        tokenizer = AutoTokenizer.from_pretrained(base)
        if model_type == "seq2seq":
            model = AutoModelForSeq2SeqLM.from_pretrained(base)
        elif model_type == "causal":
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
            model = AutoModelForCausalLM.from_pretrained(base)
        else:
            raise ValueError(f"Unknown model type for base {base}: {model_type}")
    else:
        if model_type == "causal":
            tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir)
            model = AutoModelForCausalLM.from_pretrained(checkpoint_dir)
        elif model_type == "seq2seq":
            tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir)
            model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint_dir)
        else:
            raise ValueError("When using --checkpoint_dir, you must specify --model_type (causal or seq2seq)")
    return model, tokenizer, model_type


In [5]:
def get_output_path(base=None, checkpoint_dir=None):
    os.makedirs("predictions", exist_ok=True)
    if base:
        checkpoint_name = base.replace("/", "_").replace("\\", "_")
        filename = f"{checkpoint_name}_predictions.json"
    else:
        checkpoint_dir = os.path.normpath(checkpoint_dir)
        parts = checkpoint_dir.split(os.sep)
        checkpoint_name = "_".join(parts[-2:]) if len(parts) >= 2 else parts[-1]
        checkpoint_name = checkpoint_name.replace("/", "_").replace("\\", "_")
        filename = f"{checkpoint_name}_predictions.json"

    return os.path.join("predictions", filename)


In [6]:
def generate_predictions(model, tokenizer, sources, model_type, max_len=128, batch_size=4):
    preds = []
    device = model.device
    for i in tqdm(range(0, len(sources), batch_size)):
        batch = sources[i:i + batch_size]
        if model_type == "seq2seq":
            inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True).to(device)
            with torch.no_grad():
                outputs = model.generate(**inputs, max_new_tokens=max_len)
            preds.extend(tokenizer.batch_decode(outputs, skip_special_tokens=True))
        else:  # causal (e.g., Alif / LLaMA)
            inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True).to(device)
            with torch.no_grad():
                outputs = model.generate(**inputs, max_new_tokens=max_len, do_sample=False)
            decoded = [tokenizer.decode(o, skip_special_tokens=True) for o in outputs]
            preds.extend([out.replace(inp, "").strip() for inp, out in zip(batch, decoded)])
    return preds


In [ ]:
def run_experiments(arg_list):
    for args in arg_list:
        print("\nRunning experiment:", args)

        output_path = get_output_path(args.get("base"), args.get("checkpoint_dir"))
        print(f"Output will be saved to: {output_path}")

        # Load dataset
        incorrect, correct = load_data(args["data_file"])

        # Load model
        model, tokenizer, model_type = load_model_and_tokenizer(
            base=args.get("base"),
            checkpoint_dir=args.get("checkpoint_dir"),
            model_type=args.get("model_type")
        )
        model.eval()
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model.to(device)
        print(f"Using device: {device}")

        # Generate predictions
        preds = generate_predictions(model, tokenizer, incorrect, model_type)

        # Save combined results
        combined = [
            {"incorrect": inc, "correct": cor, "predicted": pred}
            for inc, cor, pred in zip(incorrect, correct, preds)
        ]
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(combined, f, indent=2, ensure_ascii=False)
        print(f"Predictions saved to {output_path}")


In [8]:
# from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
# List of runs
arg_list = [
    # --- Base Hugging Face Models ---
    # {"base": "bigscience/mt0-large", "data_file": "../models/test_data.json", "model_type": "seq2seq"},
    {"base": "facebook/nllb-200-3.3B", "data_file": "../models/test_data.json", "model_type": "seq2seq"},
    # {"base": "google/byt5-large", "data_file": "../models/test_data.json", "model_type": "seq2seq"},
    # {"base": "large-traversaal/Alif-1.0-8B-Instruct", "data_file": "../models/test_data.json", "model_type": "causal"},

    # --- Local Fine-tuned Checkpoints (Google Drive) ---
    # {"checkpoint_dir": "/content/drive/MyDrive/checkpoints/mt0_checkpoint", "model_type": "seq2seq", "data_file": "../models/test_data.json"},
    # {"checkpoint_dir": "/content/drive/MyDrive/checkpoints/byte_checkpoint", "model_type": "seq2seq", "data_file": "../models/test_data.json"},
    # {"checkpoint_dir": "/content/drive/MyDrive/checkpoints/alif_finetuned", "model_type": "causal", "data_file": "../models/test_data.json"}
]

# Run all
run_experiments(arg_list)


🚀 Running experiment: {'base': 'facebook/nllb-200-3.3B', 'data_file': '../models/test_data.json', 'model_type': 'seq2seq'}
Output will be saved to: predictions\facebook_nllb-200-3.3B_predictions.json


c:\Users\Burhanuddin\Home\gec_pipeline\.venv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Burhanuddin\.cache\huggingface\hub\models--facebook--nllb-200-3.3B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back t

In [10]:
%pip install hf_xet

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\Burhanuddin\Home\gec_pipeline\.venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [ ]:
# Evaluate BLEU & BERTScore for all predicted files

import json
import os
import evaluate

def evaluate_all_predictions(pred_dir="predictions"):
    bleu = evaluate.load("bleu")
    bertscore = evaluate.load("bertscore")

    results = []

    for filename in os.listdir(pred_dir):
        if not filename.endswith(".json"):
            continue
        
        path = os.path.join(pred_dir, filename)
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        
        preds = [d["predicted"] for d in data]
        refs = [d["correct"] for d in data]

        print(f"\nEvaluating: {filename}")
        bleu_score = bleu.compute(predictions=preds, references=refs)
        bertscore_score = bertscore.compute(predictions=preds, references=refs, lang="ur")

        bleu_val = bleu_score["bleu"] * 100
        bert_f1_mean = sum(bertscore_score["f1"]) / len(bertscore_score["f1"])

        print(f"  BLEU Score: {bleu_val:.2f}")
        print(f"  BERTScore (F1 mean): {bert_f1_mean:.4f}")

        results.append({
            "file": filename,
            "BLEU": bleu_val,
            "BERTScore_F1": bert_f1_mean
        })

    print("\nEvaluation complete.")
    return results


# Run evaluation
all_results = evaluate_all_predictions()

# Optionally, save summary
with open("predictions/evaluation_summary.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2)

print("Summary saved to predictions/evaluation_summary.json")



📘 Evaluating: checkpoint-20000_predictions.json


c:\Users\Burhanuddin\Home\gec_pipeline\.venv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Burhanuddin\.cache\huggingface\hub\models--bert-base-multilingual-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling b

  BLEU Score: 63.50
  BERTScore (F1 mean): 0.9360

✅ Evaluation complete.

📂 Summary saved to predictions/evaluation_summary.json
